## Gold Layer  

Builds business-ready analytics tables from the Silver layer: `sales_analytics` (partitioned by territory), customer lifetime value (CLTV) metrics, loyalty scores, and new/return customer classification, saved as `workspace.gold.customer_ltv`.

## Step 1: Load all tables from the Silver Layer

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

display(spark.sql(f"SHOW TABLES IN {SILVER}"))

# Load every Silver table
sales_combined_df = spark.table(f"{SILVER}.FactSalesCombined")
customer_segments = spark.table(f"{SILVER}.FactCustomerSegment")

for name, df in [("FactSalesCombined", sales_combined_df), ("FactCustomerSegment", customer_segments)]:
    print(f"{name:<22}{df.count():>10,} rows | columns: {len(df.columns)}")

database,tableName,isTemporary
silver,factcustomersegment,false
silver,factsalescombined,false


FactSalesCombined        121,253 rows | columns: 49
FactCustomerSegment       18,485 rows | columns: 6


## Step 2: Save `sales_analytics` in the Gold Layer (partitioned by SalesTerritoryKey)

In [0]:
# --- Save FactSalesCombined into the Gold layer as sales_analytics, partitioned by territory ---
sales_analytics = spark.table(f"{SILVER}.FactSalesCombined")

sales_analytics.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("SalesTerritoryKey") \
    .saveAsTable(f"{GOLD}.sales_analytics")

# Verify partitioning and row count
partitions = spark.sql(f"SHOW PARTITIONS {GOLD}.sales_analytics")
gold_sales = spark.table(f"{GOLD}.sales_analytics")
print(f"sales_analytics partitions: {partitions.count()}")
print(f"sales_analytics rows:      {gold_sales.count():,}")
display(partitions)

sales_analytics partitions: 10
sales_analytics rows:      121,253


SalesTerritoryKey
8
4
9
2
1
5
6
3
7
10


## Step 3: Customer Lifetime Value (CLTV) from FactSalesCombined

1. Total spend per customer (descending)
2. Total purchases per customer (descending)
3. Average order value per customer (+ customer with max average)
4. Loyalty score (Platinum / Gold / Silver / Bronze)

In [0]:
# --- CLTV 3.1: Total spend per customer, descending ---
customer_spend = (
    sales_combined_df
    .groupBy("CustomerKey", "Customer")
    .agg(F.sum("Sales_Amount").alias("TotalSpend"))
)
print(f"Customers with purchases: {customer_spend.count():,}")
display(customer_spend.orderBy(F.desc("TotalSpend")).limit(10))

Customers with purchases: 18,485


CustomerKey,Customer,TotalSpend
-1,[Not Applicable],8.045059610999915E7
12301,Nichole Nara,13295.38
12132,Kaitlyn Henderson,13294.269999999999
12308,Margaret He,13269.269999999999
12131,Randall Dominguez,13265.99
12300,Adriana Gonzalez,13242.699999999999
12321,Rosa Hu,13215.649999999998
12124,Brandi Gill,13195.64
12307,Brad She,13173.189999999999
12296,Francisco Sara,13164.639999999998


In [0]:
# --- CLTV 3.2: Total purchases per customer, descending ---
# A purchase = a distinct sales order (a customer may buy several product lines in one order)
customer_purchases = (
    sales_combined_df
    .groupBy("CustomerKey", "Customer")
    .agg(F.countDistinct("Sales_Order").alias("TotalPurchases"))
)
display(customer_purchases.orderBy(F.desc("TotalPurchases")).limit(10))

CustomerKey,Customer,TotalPurchases
-1,[Not Applicable],3796
11176,Mason Roberts,28
11091,Dalton Perez,28
11262,Jennifer Simmons,27
11711,Daniel Davis,27
11223,Hailey Patterson,27
11330,Ryan Thompson,27
11276,Nancy Chapman,27
11185,Ashley Henderson,27
11200,Jason Griffin,27


In [0]:
# --- CLTV 3.3: Average order value per customer ---
customer_cltv = (
    sales_combined_df
    .groupBy("CustomerKey", "Customer")
    .agg(
        F.sum("Sales_Amount").alias("TotalSpend"),
        F.countDistinct("Sales_Order").alias("TotalPurchases"),
    )
    .withColumn("AvgPurchaseValue", F.round(F.col("TotalSpend") / F.col("TotalPurchases"), 2))
)
display(customer_cltv.orderBy(F.desc("AvgPurchaseValue")).limit(10))

# Customer with the maximum average order value
max_avg = customer_cltv.orderBy(F.desc("AvgPurchaseValue")).limit(1).collect()[0]
print(f"Maximum average order value: {max_avg['Customer']} (${max_avg['AvgPurchaseValue']:,.2f})")

# NOTE: '[Not Applicable]' (CustomerKey = -1) is the placeholder that aggregates all
# reseller-channel sales, so also report the top INDIVIDUAL customer
max_avg_real = customer_cltv.filter(F.col("CustomerKey") != -1) \
    .orderBy(F.desc("AvgPurchaseValue")).limit(1).collect()[0]
print(f"Max average order value among individual customers: {max_avg_real['Customer']} (${max_avg_real['AvgPurchaseValue']:,.2f})")

CustomerKey,Customer,TotalSpend,TotalPurchases,AvgPurchaseValue
-1,[Not Applicable],8.045059610999992E7,3796,21193.52
28643,Xavier Gray,3578.27,1,3578.27
28586,Gabriel Diaz,3578.27,1,3578.27
28197,Kayla Rodriguez,3578.27,1,3578.27
28150,Connor Chen,3578.27,1,3578.27
28162,Bruce Ward,3578.27,1,3578.27
29010,Jennifer Long,3578.27,1,3578.27
28997,Hector Carlson,3578.27,1,3578.27
29170,Alexandra Adams,3578.27,1,3578.27
29156,Ashley Barnes,3578.27,1,3578.27


Maximum average order value: [Not Applicable] ($21,193.52)
Max average order value among individual customers: Connor Chen ($3,578.27)


In [0]:
# --- CLTV 3.4: Loyalty score based on TotalSpend ---
#   TotalSpend > 60000 -> Platinum | > 50000 -> Gold | > 20000 -> Silver | else Bronze
customer_cltv = customer_cltv.withColumn(
    "LoyaltyScore",
    F.when(F.col("TotalSpend") > 60000, "Platinum")
     .when(F.col("TotalSpend") > 50000, "Gold")
     .when(F.col("TotalSpend") > 20000, "Silver")
     .otherwise("Bronze"),
)
display(
    customer_cltv
    .groupBy("LoyaltyScore")
    .agg(F.count("*").alias("customers"), F.sum("TotalSpend").alias("total_spend"))
    .orderBy(F.desc("total_spend"))
)

LoyaltyScore,customers,total_spend
Platinum,1,8.045059610999915E7
Bronze,18484,2.9358677889999494E7


## Step 4: New vs Return Customers

A **return customer** has ordered more than once (more than one distinct sales order).

In [0]:
# --- New vs return customers ---
customer_cltv = customer_cltv.withColumn(
    "CustomerType",
    F.when(F.col("TotalPurchases") > 1, "Return Customer").otherwise("New Customer"),
)

display(customer_cltv.groupBy("CustomerType").agg(
    F.count("*").alias("customers"),
    F.sum("TotalPurchases").alias("total_orders"),
    F.sum("TotalSpend").alias("total_spend"),
))

# Top return customers by number of orders
display(
    customer_cltv
    .filter(F.col("CustomerType") == "Return Customer")
    .select("Customer", "TotalPurchases", "TotalSpend")
    .orderBy(F.desc("TotalPurchases"))
    .limit(10)
)

CustomerType,customers,total_orders,total_spend
New Customer,11619,11619,6747965.650000549
Return Customer,6866,19836,1.0306130834999892E8


Customer,TotalPurchases,TotalSpend
[Not Applicable],3796,8.045059610999992E7
Mason Roberts,28,1320.0100000000002
Dalton Perez,28,1189.3300000000002
Jennifer Simmons,27,1126.8200000000004
Daniel Davis,27,1103.78
Hailey Patterson,27,1205.3800000000003
Ryan Thompson,27,1136.3500000000004
Nancy Chapman,27,988.3200000000003
Ashley Henderson,27,1616.3100000000002
Jason Griffin,27,1463.3799999999999


## Step 5: Save `customer_ltv` to the Gold Layer

In [0]:
# --- Save the CLTV dataset as a Gold Delta table ---
customer_ltv = customer_cltv.select(
    "CustomerKey", "Customer", "TotalSpend", "TotalPurchases",
    "AvgPurchaseValue", "LoyaltyScore", "CustomerType",
)
customer_ltv.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.customer_ltv")


In [0]:
# --- Final validation: record count + current user ---
customer_ltv_df = spark.table(f"{GOLD}.customer_ltv")
print(f"Total Customers in LTV Table: {customer_ltv_df.count()}")

print(dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get())

Total Customers in LTV Table: 18485
hemanth.appari@maqsoftware.com
